# Load Test Data Visualization (Seaborn)

横轴: 真实重量(kg)
纵轴: 估计的 Z 轴力(N, `ext_wrench[2]`)

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

sns.set_theme(style="whitegrid")

data_path = Path("../data/load_test_data.json")
with data_path.open("r", encoding="utf-8") as f:
    raw = json.load(f)

rows = []
for load_str, trials in raw.items():
    load_kg = float(load_str)
    for trial_idx, trial_samples in enumerate(trials, start=1):
        for sample_idx, item in enumerate(trial_samples, start=1):
            rows.append({
                "real_weight_kg": load_kg,
                "trial": trial_idx,
                "sample": sample_idx,
                "estimated_fz_n": float(item["ext_wrench"][2]),
            })

df = pd.DataFrame(rows)
df = df.sort_values(["real_weight_kg", "trial", "sample"]).reset_index(drop=True)

display(df.head())
display(df.groupby("real_weight_kg")["estimated_fz_n"].agg(["count", "mean", "std"]))

In [ ]:
# 1. 散点图
plt.figure(figsize=(8, 5))
sns.scatterplot(
    data=df,
    x="real_weight_kg",
    y="estimated_fz_n",
    s=45,
    alpha=0.75,
)
plt.xlabel("Real Weight (kg)")
plt.ylabel("Estimated Z-axis Force (N)")
plt.title("Scatter: Weight vs Estimated Z Force")
plt.tight_layout()
plt.show()

In [ ]:
# 2. 箱线图
order = sorted(df["real_weight_kg"].unique())
plt.figure(figsize=(8, 5))
sns.boxplot(
    data=df,
    x="real_weight_kg",
    y="estimated_fz_n",
    order=order,
    width=0.6,
)
plt.xlabel("Real Weight (kg)")
plt.ylabel("Estimated Z-axis Force (N)")
plt.title("Box Plot: Estimated Z Force by Weight")
plt.tight_layout()
plt.show()

In [ ]:
# 3. 带误差线的柱状图
plt.figure(figsize=(8, 5))
sns.barplot(
    data=df,
    x="real_weight_kg",
    y="estimated_fz_n",
    order=sorted(df["real_weight_kg"].unique()),
    estimator="mean",
    errorbar="sd",
    capsize=0.15,
)
plt.xlabel("Real Weight (kg)")
plt.ylabel("Estimated Z-axis Force (N)")
plt.title("Bar Plot with SD Error Bars")
plt.tight_layout()
plt.show()

In [ ]:
# 4. 带标准差的折线图(趋势图)
plt.figure(figsize=(8, 5))
sns.lineplot(
    data=df,
    x="real_weight_kg",
    y="estimated_fz_n",
    estimator="mean",
    errorbar="sd",
    marker="o",
    linewidth=2.0,
)
plt.xlabel("Real Weight (kg)")
plt.ylabel("Estimated Z-axis Force (N)")
plt.title("Trend Line with SD Band")
plt.tight_layout()
plt.show()

In [ ]:
# 5. 散点 + 线性拟合线
x = df["real_weight_kg"].to_numpy()
y = df["estimated_fz_n"].to_numpy()
slope, intercept = np.polyfit(x, y, 1)
x_fit = np.linspace(x.min(), x.max(), 200)
y_fit = slope * x_fit + intercept

plt.figure(figsize=(8, 5))
sns.scatterplot(data=df, x="real_weight_kg", y="estimated_fz_n", s=35, alpha=0.6, label="Samples")
sns.lineplot(x=x_fit, y=y_fit, color="tab:red", linewidth=2.2, label=f"Fit: y={slope:.3f}x+{intercept:.3f}")
plt.xlabel("Real Weight (kg)")
plt.ylabel("Estimated Z-axis Force (N)")
plt.title("Scatter + Linear Fit")
plt.tight_layout()
plt.show()

print(f"Linear fit: estimated_fz_n = {slope:.6f} * real_weight_kg + {intercept:.6f}")